<a href="https://colab.research.google.com/github/MLfinal/Walmart-Recruiting---Store-Sales-Forecasting/blob/dev/models/deep_learning/DLinear/dlinear_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DLinear inference

This notebook generates Kaggle test predictions from the best DLinear model stored in W&B.

Default model choice:

```text
manual_v1 = DLinear + Store-Dept series_bias
validation WMAE = 1506.28
```

The tuned model is available as a fallback, but it did not beat manual v1.

In [ ]:
%pip install -q "torch>=2.3,<3" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "kaggle>=1.7,<2"

In [ ]:
from __future__ import annotations

import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import wandb

pd.set_option("display.max_columns", 100)
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
})

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Not running in Colab or Drive unavailable: {exc}")

In [ ]:
CONFIG = {
    "data_dir": "/content/drive/MyDrive/walmart_competition_data",
    "output_dir": "/content/drive/MyDrive/walmart_competition_inference/dlinear",
    "download_dir": "/content/artifacts/dlinear_model",
    "wandb_entity": "kende23-n-a",
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "model_choice": "manual_v1",  # "manual_v1" or "tuned_best"
    "manual_v1_artifact_uri": "kende23-n-a/Walmart-Recruiting---Store-Sales-Forecasting/dlinear-v1-104w-series-calibration:experiment-v1",
    "tuned_best_artifact_uri": "kende23-n-a/Walmart-Recruiting---Store-Sales-Forecasting/dlinear-tuned-best-series-calibration:tuned-best",
    "registry_target": "wandb-registry-model/Walmart_DLinear_Model",
    "submission_artifact_name": "dlinear-kaggle-submission",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "submit_to_kaggle": False,
    "kaggle_competition": "walmart-recruiting-store-sales-forecasting",
}

DATA_DIR = Path(CONFIG["data_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
DOWNLOAD_DIR = Path(CONFIG["download_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ARTIFACT_URI = CONFIG[f"{CONFIG['model_choice']}_artifact_uri"]
CONFIG

## Model definition

This class is intentionally compatible with the v1 and tuned-best checkpoints.

In [ ]:
class MovingAverage(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.kernel_size = int(kernel_size)
        self.avg = nn.AvgPool1d(kernel_size=self.kernel_size, stride=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pad_left = (self.kernel_size - 1) // 2
        pad_right = self.kernel_size - 1 - pad_left
        front = x[:, :1, :].repeat(1, pad_left, 1)
        end = x[:, -1:, :].repeat(1, pad_right, 1)
        x_pad = torch.cat([front, x, end], dim=1)
        return self.avg(x_pad.permute(0, 2, 1)).permute(0, 2, 1)


class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.moving_average = MovingAverage(kernel_size)

    def forward(self, x: torch.Tensor):
        trend = self.moving_average(x)
        seasonal = x - trend
        return seasonal, trend


class DLinearSeriesCalibration(nn.Module):
    def __init__(self, seq_len: int, pred_len: int, n_series: int, moving_avg_kernel: int = 25):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.decomposition = SeriesDecomposition(moving_avg_kernel)
        self.linear_seasonal = nn.Linear(self.seq_len, self.pred_len)
        self.linear_trend = nn.Linear(self.seq_len, self.pred_len)
        self.series_bias = nn.Embedding(n_series, self.pred_len)

    def forward(self, x: torch.Tensor, series_idx: torch.Tensor) -> torch.Tensor:
        seasonal, trend = self.decomposition(x)
        seasonal = seasonal.permute(0, 2, 1)
        trend = trend.permute(0, 2, 1)
        out = self.linear_seasonal(seasonal) + self.linear_trend(trend)
        out = out.permute(0, 2, 1).squeeze(-1)
        return out + self.series_bias(series_idx)

## Load raw data

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])

required_train = {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}
required_test = {"Store", "Dept", "Date", "IsHoliday"}
missing_train = sorted(required_train.difference(train_raw.columns))
missing_test = sorted(required_test.difference(test_raw.columns))
if missing_train or missing_test:
    raise ValueError({"missing_train": missing_train, "missing_test": missing_test})

train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print("train", train_raw.shape, train_raw["Date"].min(), train_raw["Date"].max())
print("test", test_raw.shape, test_raw["Date"].min(), test_raw["Date"].max())

## Start W&B inference run and download model artifact

In [ ]:
run = wandb.init(
    project=CONFIG["wandb_project"],
    entity=CONFIG["wandb_entity"],
    job_type="dlinear_inference",
    name=f"dlinear_inference_{CONFIG['model_choice']}",
    config=CONFIG,
)

model_artifact = run.use_artifact(MODEL_ARTIFACT_URI)
artifact_dir = Path(model_artifact.download(root=str(DOWNLOAD_DIR)))
print("Downloaded artifact:", model_artifact.name, "->", artifact_dir)
print("Artifact files:", [p.name for p in artifact_dir.iterdir()])

checkpoint_candidates = sorted(artifact_dir.glob("*.pt"))
if not checkpoint_candidates:
    raise FileNotFoundError(f"No .pt checkpoint found in {artifact_dir}")
checkpoint_path = checkpoint_candidates[0]
print("Using checkpoint:", checkpoint_path)

## Load checkpoint and reconstruct input panel

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location="cpu")
checkpoint_config = checkpoint.get("config", {})
best_params = checkpoint.get("best_params", {})
series_index_raw = checkpoint.get("series_index")
if series_index_raw is None:
    raise KeyError("Checkpoint does not contain series_index; cannot align Store-Dept predictions safely.")

series_index = pd.MultiIndex.from_tuples([tuple(x) for x in series_index_raw], names=["Store", "Dept"])

# Prefer the checkpoint tensor shapes over config values. This makes inference robust even if an older run name/config
# says 104w while the saved DLinear weights were trained with a 52w input window.
state_dict = checkpoint["model_state_dict"]
if "linear_trend.weight" in state_dict:
    inferred_pred_len, inferred_input_weeks = state_dict["linear_trend.weight"].shape
else:
    inferred_input_weeks = int(best_params.get("input_weeks", checkpoint_config.get("input_weeks", 52)))
    inferred_pred_len = int(checkpoint_config.get("validation_weeks", best_params.get("validation_weeks", 39)))

input_weeks = int(inferred_input_weeks)
pred_len = int(inferred_pred_len)
moving_avg_kernel = int(best_params.get("moving_avg_kernel", checkpoint_config.get("moving_avg_kernel", 25)))

all_train_dates = pd.Index(sorted(train_raw["Date"].unique()), name="Date")
test_dates = pd.Index(sorted(test_raw["Date"].unique()), name="Date")
if len(test_dates) != pred_len:
    raise ValueError(f"Checkpoint predicts {pred_len} weeks, but test has {len(test_dates)} dates.")
if len(all_train_dates) < input_weeks:
    raise ValueError(f"Need at least {input_weeks} train dates, got {len(all_train_dates)}")

sales_panel = (
    train_raw.pivot_table(index=["Store", "Dept"], columns="Date", values="Weekly_Sales", aggfunc="sum")
    .reindex(index=series_index, columns=all_train_dates)
    .fillna(0.0)
)

print({
    "model_choice": CONFIG["model_choice"],
    "artifact": MODEL_ARTIFACT_URI,
    "checkpoint": str(checkpoint_path),
    "n_series": len(series_index),
    "input_weeks": input_weeks,
    "pred_len": pred_len,
    "moving_avg_kernel": moving_avg_kernel,
    "test_dates": (str(test_dates.min().date()), str(test_dates.max().date())),
})

## Predict test horizon

In [ ]:
device = CONFIG["device"]
model = DLinearSeriesCalibration(
    seq_len=input_weeks,
    pred_len=pred_len,
    n_series=len(series_index),
    moving_avg_kernel=moving_avg_kernel,
)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(device)
model.eval()

values = sales_panel.to_numpy(dtype=np.float32)
x_last = values[:, -input_weeks:]
means = x_last.mean(axis=1, keepdims=True).astype(np.float32)
stds = x_last.std(axis=1, keepdims=True).astype(np.float32)
stds = np.maximum(stds, 1.0)
x_norm = ((x_last - means) / stds).astype(np.float32)

batch_size = 1024
pred_batches = []
with torch.no_grad():
    for start in range(0, len(series_index), batch_size):
        end = min(start + batch_size, len(series_index))
        x = torch.from_numpy(x_norm[start:end, :, None]).to(device)
        series_idx = torch.arange(start, end, dtype=torch.long, device=device)
        pred_norm = model(x, series_idx).cpu().numpy()
        pred = pred_norm * stds[start:end] + means[start:end]
        pred_batches.append(pred)

test_pred = np.clip(np.concatenate(pred_batches, axis=0), 0.0, None)
print({
    "prediction_shape": test_pred.shape,
    "prediction_min": float(np.min(test_pred)),
    "prediction_mean": float(np.mean(test_pred)),
    "prediction_max": float(np.max(test_pred)),
})

## Build Kaggle submission

In [ ]:
pred_lookup = {}
for row_idx, key in enumerate(series_index):
    store, dept = int(key[0]), int(key[1])
    for horizon_idx, date in enumerate(test_dates):
        pred_lookup[(store, dept, pd.Timestamp(date))] = float(test_pred[row_idx, horizon_idx])

submission = test_raw[["Store", "Dept", "Date"]].copy()
submission["Weekly_Sales"] = [
    pred_lookup.get((int(r.Store), int(r.Dept), pd.Timestamp(r.Date)), 0.0)
    for r in submission.itertuples(index=False)
]
submission.insert(
    0,
    "Id",
    submission["Store"].astype(str) + "_" + submission["Dept"].astype(str) + "_" + submission["Date"].dt.strftime("%Y-%m-%d"),
)
submission = submission[["Id", "Weekly_Sales"]]

missing_keys = sum(
    (int(r.Store), int(r.Dept), pd.Timestamp(r.Date)) not in pred_lookup
    for r in test_raw.itertuples(index=False)
)

submission_path = OUTPUT_DIR / f"submission_dlinear_{CONFIG['model_choice']}.csv"
submission.to_csv(submission_path, index=False)
print({
    "submission_path": str(submission_path),
    "rows": len(submission),
    "missing_store_dept_date_predictions_filled_zero": missing_keys,
})
display(submission.head())

## Log inference diagnostics, submission, and link model to W&B Registry

In [ ]:
manifest = {
    "model_choice": CONFIG["model_choice"],
    "model_artifact_uri": MODEL_ARTIFACT_URI,
    "model_artifact_name": model_artifact.name,
    "checkpoint_path": str(checkpoint_path),
    "input_weeks": input_weeks,
    "pred_len": pred_len,
    "moving_avg_kernel": moving_avg_kernel,
    "n_series": int(len(series_index)),
    "test_rows": int(len(test_raw)),
    "submission_rows": int(len(submission)),
    "missing_predictions_filled_zero": int(missing_keys),
    "prediction_min": float(submission["Weekly_Sales"].min()),
    "prediction_mean": float(submission["Weekly_Sales"].mean()),
    "prediction_max": float(submission["Weekly_Sales"].max()),
}
manifest_path = OUTPUT_DIR / f"dlinear_inference_manifest_{CONFIG['model_choice']}.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(submission["Weekly_Sales"], bins=80)
ax.set_title(f"DLinear {CONFIG['model_choice']} test prediction distribution")
ax.set_xlabel("Weekly_Sales")
ax.set_ylabel("count")
plt.tight_layout()
hist_path = OUTPUT_DIR / f"dlinear_prediction_hist_{CONFIG['model_choice']}.png"
fig.savefig(hist_path, dpi=160)
plt.show()

wandb.log({
    "inference/prediction_min": manifest["prediction_min"],
    "inference/prediction_mean": manifest["prediction_mean"],
    "inference/prediction_max": manifest["prediction_max"],
    "inference/submission_rows": manifest["submission_rows"],
    "inference/missing_predictions_filled_zero": manifest["missing_predictions_filled_zero"],
    "inference/submission_preview": wandb.Table(dataframe=submission.head(2000)),
    "inference/prediction_histogram": wandb.Image(str(hist_path)),
})

submission_artifact = wandb.Artifact(CONFIG["submission_artifact_name"], type="submission")
submission_artifact.add_file(str(submission_path))
submission_artifact.add_file(str(manifest_path))
submission_artifact.add_file(str(hist_path))
run.log_artifact(submission_artifact, aliases=[CONFIG["model_choice"], "latest"])

# Make the selected DLinear model stand out in W&B Model Registry.
try:
    run.link_artifact(model_artifact, target_path=CONFIG["registry_target"], aliases=["champion", CONFIG["model_choice"], "latest"])
    print("Linked model artifact to registry:", CONFIG["registry_target"])
except Exception as exc:
    print("Registry link skipped or failed:", repr(exc))

manifest

## Optional Kaggle submission upload

In [ ]:
if CONFIG["submit_to_kaggle"]:
    try:
        from google.colab import userdata
        import os
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception as exc:
        print("Could not load Kaggle secrets from Colab userdata:", exc)

    import subprocess
    cmd = [
        "kaggle", "competitions", "submit",
        "-c", CONFIG["kaggle_competition"],
        "-f", str(submission_path),
        "-m", f"DLinear {CONFIG['model_choice']} W&B artifact inference",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Kaggle upload disabled. Set CONFIG['submit_to_kaggle'] = True to upload.")

In [ ]:
run.finish()